# HISTOPANTUM colon: 5-fold cross-validation, 80/20 case-disjoint

This notebook replaces exp-6/exp-7's single 28/6/6 case-disjoint split with a
single **80/20 case-disjoint hold-out** (32 cv-pool cases / 8 test cases) and
a **5-fold `StratifiedGroupKFold` cross-validation** over the 80% pool,
grouped by `case_id` and stratified by each case's tumour fraction.

It reuses, unchanged, exp-6's ResNet50 protocol and exp-7's MobileNetV2
protocol: same augmentation, same preprocessing, same two-phase training
(frozen head, then partial fine-tuning), same checkpoint-by-`val_loss`
selection rule. The only thing that changes is which cases fall into which
partition, and that there are now 5 CV folds instead of one small validation
set.

**It does not touch exp-6 or exp-7's own case-to-partition mapping.** The new
80/20 assignment lives in `outputs/cv_split_assignment.csv`, built locally
(no GPU, no images) by `build_cv_split.py` from case-level patch counts
already exported by exp-6. See that script and the exp-8 `README.md` for how
it was derived and why it is deterministic.

## What this notebook produces, per architecture (ResNet50, MobileNetV2)

- `outputs/{architecture}_fold{0..4}_predictions.csv`: out-of-fold per-patch
  probabilities for the 32 cv-pool cases (each case appears in exactly one
  fold's file, from whichever fold held it out).
- `outputs/{architecture}_fold{0..4}_case_metrics.csv`: per-case accuracy,
  balanced accuracy, F1, ROC-AUC for that fold's held-out cases.
- `outputs/{architecture}_test_predictions.csv` /
  `outputs/{architecture}_test_metrics.json`: **one** evaluation, on the
  8-case held-out test group, using the fold-0-trained checkpoint as the
  prespecified deployment candidate (see "Which checkpoint is the final
  model?" below) -- same JSON schema as exp-6/exp-7's `*_test_metrics.json`.
- `outputs/{architecture}_model_manifest.json`: identity and SHA-256 of the
  checkpoint evaluated on test.

Cross-fold aggregation (mean +/- std across the 5 folds) is **not** computed
here -- it only needs the CSVs above, so it is computed locally afterward by
`experiments/exp-8/summarize_results.py` (no GPU needed). Copy this
notebook's `outputs/` contents into `experiments/exp-8/outputs/` in the repo
when the run finishes.

## Which checkpoint is the final model?

Fold 0's trained model is **prespecified**, before looking at any fold's
validation result, as the model evaluated once on the true 8-case held-out
test set. This is not "pick the best fold": the choice of fold 0 is fixed
here, in this cell, before training starts. The other 4 folds only ever
contribute to the cross-validation *estimate* (mean +/- std), never to
choosing which model touches the test set. This mirrors exp-6/exp-7's own
rule of selecting a checkpoint by validation loss and evaluating test exactly
once.

In [ ]:
!unzip /content/histopantum.zip -d /content/histopantum

Streaming output truncated to the last 5000 lines.
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16384_15360.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16384_16384.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16384_27136.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16384_27648.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16896_13824.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16896_28160.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_17408_13312.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_17408_14848.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_17408_15872.jpg  
  inflating: /content/histopantum/histop

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from tensorflow import keras

# ---------------------------------------------------------------------------
# Parametrized paths -- edit these for your Colab session, nothing else in
# this notebook needs to change.
# ---------------------------------------------------------------------------
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 15
HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
FINAL_CANDIDATE_FOLD = 0  # prespecified: fold 0's model is evaluated on test

DATASET_ROOT = Path(os.environ.get('HISTOPANTUM_COLON_ROOT', '/content/histopantum/histopantum/colon'))
CV_SPLIT_CSV = Path(os.environ.get('HISTOPANTUM_CV_SPLIT_CSV', '/content/cv_split_assignment.csv'))
OUTPUT_DIR = Path('/content/exp8_outputs') if Path('/content').exists() else Path('outputs')

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f'Deterministic TensorFlow operations unavailable: {exc}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'python': sys.version, 'tensorflow': tf.__version__, 'sklearn': sklearn.__version__})

{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'tensorflow': '2.20.0', 'sklearn': '1.6.1'}


## Dataset and split assignment

In [ ]:
# In Colab: mount Drive or upload the dataset zip before this cell, then set
# HISTOPANTUM_COLON_ROOT above if it does not match the default candidates.
candidates = [
    DATASET_ROOT,
    Path('/content/histopantum/colon'),
    Path('/content/colon'),
    Path('data/histopantum/colon'),
]
DATASET_ROOT = next((path for path in candidates if path.is_dir()), DATASET_ROOT)
required = [DATASET_ROOT / 'non-tumour', DATASET_ROOT / 'tumour']
assert all(path.is_dir() for path in required), (
    f'Dataset not found at {DATASET_ROOT}. Expected non-tumour/ and tumour/.'
)
assert CV_SPLIT_CSV.is_file(), (
    f'{CV_SPLIT_CSV} not found. Upload experiments/exp-8/outputs/cv_split_assignment.csv '
    'to Colab (or point HISTOPANTUM_CV_SPLIT_CSV at it) before running this notebook.'
)
cv_split = pd.read_csv(CV_SPLIT_CSV)
assert set(cv_split['group']) == {'test', 'fold0', 'fold1', 'fold2', 'fold3', 'fold4'}
assert len(cv_split) == 40
print('Dataset root:', DATASET_ROOT.resolve())
print(cv_split.groupby('group').agg(cases=('case_id', 'size'), patches=('patches', 'sum')))

Dataset root: /content/histopantum/histopantum/colon
       cases  patches
group                
fold0      7     5635
fold1      7     5558
fold2      6     4447
fold3      6     2679
fold4      6     3482
test       8     5447


In [ ]:
def parse_patch(path: Path, label: int) -> dict:
    """Parse a HISTOPANTUM filename into case, slide, and coordinates.

    Identical to exp-6's parser -- only the partition assignment changes.
    """
    stem = path.stem
    parts = stem.rsplit('_', 2)
    if len(parts) != 3 or not parts[1].isdigit() or not parts[2].isdigit():
        raise ValueError(f'Unexpected patch filename: {path.name}')
    slide_id, x, y = parts
    case_parts = slide_id.split('-')
    if len(case_parts) < 3 or case_parts[0] != 'TCGA':
        raise ValueError(f'Unexpected TCGA slide identifier: {slide_id}')
    return {
        'path': str(path.resolve()), 'relative_path': path.relative_to(DATASET_ROOT).as_posix(),
        'label': label, 'class_name': 'tumour' if label else 'non-tumour',
        'case_id': '-'.join(case_parts[:3]), 'slide_id': slide_id,
        'x': int(x), 'y': int(y),
    }

records = []
for class_name, label in [('non-tumour', 0), ('tumour', 1)]:
    paths = sorted((DATASET_ROOT / class_name).glob('*.jpg'))
    assert paths, f'No JPEG files found for {class_name}'
    records.extend(parse_patch(path, label) for path in paths)
manifest = pd.DataFrame(records)
assert not manifest['relative_path'].duplicated().any()
assert set(manifest['label']) == {0, 1}

group_of = cv_split.set_index('case_id')['group']
assert set(manifest['case_id']) == set(group_of.index), 'Manifest cases do not match cv_split_assignment.csv'
manifest['group'] = manifest['case_id'].map(group_of)
assert not manifest['group'].isna().any()

# Cross-check per-case patch/tumour counts against cv_split_assignment.csv --
# if these disagree, the dataset on this machine is not the one the split was
# built from, and nothing downstream should be trusted.
recount = manifest.groupby('case_id')['label'].agg(patches='size', tumour='sum')
expected = cv_split.set_index('case_id')[['patches', 'tumour']]
assert recount[['patches', 'tumour']].equals(expected.loc[recount.index]), (
    'Per-case patch/tumour counts differ from cv_split_assignment.csv -- '
    'wrong dataset copy or a corrupted archive.'
)
print('Cases:', manifest['case_id'].nunique(), 'Patches:', len(manifest))

Cases: 40 Patches: 27248


## Shared data pipeline, augmentation, and architecture builders

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path: tf.Tensor, label: tf.Tensor):
    """Decode one JPEG as a fixed-shape float32 RGB tensor."""
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.float32)

def make_dataset(frame: pd.DataFrame, training: bool) -> tf.data.Dataset:
    """Create a finite deterministic dataset; augmentation lives in the model."""
    dataset = tf.data.Dataset.from_tensor_slices((frame['path'].to_numpy(), frame['label'].to_numpy()))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def compile_model(target: keras.Model, learning_rate: float) -> None:
    """Compile the binary classifier with reproducible metrics."""
    target.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='roc_auc')],
    )

def make_callbacks(checkpoint_path: Path) -> list:
    return [
        keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=0),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=0),
    ]

In [ ]:
def build_resnet50():
    """ResNet50 backbone + head, identical to exp-6's model cell.

    Returns (model, custom_objects, backbone_layer_name, fine_tune_rule).
    ``fine_tune_rule`` is consumed by ``unfreeze_backbone`` below and mirrors
    each source notebook's own unfreezing logic exactly.
    """
    augmentation = keras.Sequential([
        keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
        keras.layers.RandomRotation(0.25, fill_mode='reflect', seed=SEED),
        keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=SEED),
        keras.layers.RandomContrast(0.10, seed=SEED),
    ], name='training_augmentation')
    backbone = keras.applications.ResNet50(
        include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3), pooling='avg'
    )
    backbone.trainable = False
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = augmentation(inputs)
    x = keras.layers.Lambda(keras.applications.resnet50.preprocess_input, name='caffe_preprocessing')(x)
    x = backbone(x, training=False)
    x = keras.layers.Dropout(0.30, seed=SEED)(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', name='tumour_probability')(x)
    model = keras.Model(inputs, outputs, name='histopantum_crc_resnet50')
    custom_objects = {'preprocess_input': keras.applications.resnet50.preprocess_input}
    # Captured from the live object rather than hardcoded, so a Keras version
    # difference in Colab can't silently break model.get_layer(backbone_name).
    backbone_name = backbone.name
    fine_tune_rule = ('prefix', 'conv5_')  # exp-6: unfreeze every conv5_* layer
    return model, custom_objects, backbone_name, fine_tune_rule


def build_mobilenetv2():
    """MobileNetV2 backbone + head, identical to exp-7's model cell."""
    augmentation = keras.Sequential([
        keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
        keras.layers.RandomRotation(0.25, fill_mode='reflect', seed=SEED),
        keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=SEED),
        keras.layers.RandomContrast(0.10, seed=SEED),
    ], name='training_augmentation')
    backbone = keras.applications.MobileNetV2(
        include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3), pooling='avg'
    )
    backbone.trainable = False
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = augmentation(inputs)
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name='mobilenet_v2_preprocessing')(x)
    x = backbone(x, training=False)
    x = keras.layers.Dropout(0.30, seed=SEED)(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', name='tumour_probability')(x)
    model = keras.Model(inputs, outputs, name='histopantum_crc_mobilenetv2')
    custom_objects = {}
    # exp-7's backbone submodel keeps Keras's default MobileNetV2 layer name.
    backbone_name = backbone.name
    fine_tune_rule = ('from_layer', 'block_14_expand')  # exp-7: unfreeze block_14_expand onward
    return model, custom_objects, backbone_name, fine_tune_rule


ARCHITECTURE_BUILDERS = {'resnet50': build_resnet50, 'mobilenetv2': build_mobilenetv2}


def unfreeze_backbone(model: keras.Model, backbone_name: str, fine_tune_rule: tuple) -> None:
    """Unfreeze the fine-tuning layers of ``model``'s named backbone
    submodel in place, reproducing exp-6/exp-7's own unfreezing logic
    (BatchNorm always stays frozen).
    """
    backbone = model.get_layer(backbone_name)
    backbone.trainable = True
    rule_type, rule_value = fine_tune_rule
    if rule_type == 'prefix':
        for layer in backbone.layers:
            layer.trainable = layer.name.startswith(rule_value) and not isinstance(layer, keras.layers.BatchNormalization)
    elif rule_type == 'from_layer':
        start = next(i for i, layer in enumerate(backbone.layers) if layer.name == rule_value)
        for index, layer in enumerate(backbone.layers):
            layer.trainable = index >= start and not isinstance(layer, keras.layers.BatchNormalization)
    else:
        raise ValueError(f'Unknown fine_tune_rule type: {rule_type}')

    trainable_backbone_layers = [layer.name for layer in backbone.layers if layer.trainable]
    assert trainable_backbone_layers, 'No backbone layer was unfrozen -- check fine_tune_rule.'
    assert not any(layer.trainable for layer in backbone.layers if isinstance(layer, keras.layers.BatchNormalization))

## Generic two-phase training routine (one fold, one architecture)

In [ ]:
def train_one_fold(architecture: str, fold_id: int, manifest: pd.DataFrame, output_dir: Path) -> dict:
    """Train the frozen head, then fine-tune, on fold_id's complement, and
    export out-of-fold predictions for fold_id's held-out cases.

    Returns a dict with the selected checkpoint path and its validation
    metrics, so the caller can decide whether this fold's model should also
    become the final test-evaluated candidate (see FINAL_CANDIDATE_FOLD).
    """
    fold_name = f'fold{fold_id}'
    train_frame = manifest.loc[
        (manifest['group'] != 'test') & (manifest['group'] != fold_name)
    ].reset_index(drop=True)
    val_frame = manifest.loc[manifest['group'] == fold_name].reset_index(drop=True)
    assert set(train_frame['case_id']).isdisjoint(val_frame['case_id'])
    print(f'[{architecture}/{fold_name}] train cases={train_frame["case_id"].nunique()} '
          f'({len(train_frame)} patches), val cases={val_frame["case_id"].nunique()} '
          f'({len(val_frame)} patches)')

    train_ds = make_dataset(train_frame, training=True)
    val_ds = make_dataset(val_frame, training=False)

    build_fn = ARCHITECTURE_BUILDERS[architecture]
    model, custom_objects, backbone_name, fine_tune_rule = build_fn()
    compile_model(model, HEAD_LEARNING_RATE)

    frozen_ckpt = output_dir / f'{architecture}_{fold_name}_frozen.keras'
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
              callbacks=make_callbacks(frozen_ckpt), verbose=2)
    assert frozen_ckpt.is_file()

    model = keras.models.load_model(frozen_ckpt, custom_objects=custom_objects)
    unfreeze_backbone(model, backbone_name, fine_tune_rule)
    compile_model(model, FINE_TUNE_LEARNING_RATE)

    fine_ckpt = output_dir / f'{architecture}_{fold_name}_fine_tuned.keras'
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
              callbacks=make_callbacks(fine_ckpt), verbose=2)
    assert fine_ckpt.is_file()

    checkpoints = {'frozen': frozen_ckpt, 'fine_tuned': fine_ckpt}
    validation_results = {}
    for name, ckpt in checkpoints.items():
        candidate = keras.models.load_model(ckpt, custom_objects=custom_objects)
        validation_results[name] = candidate.evaluate(val_ds, return_dict=True, verbose=0)
    selected_name = min(validation_results, key=lambda name: validation_results[name]['loss'])
    selected_ckpt = checkpoints[selected_name]
    print(f'[{architecture}/{fold_name}] selected={selected_name} val_loss={validation_results[selected_name]["loss"]:.4f}')

    selected_model = keras.models.load_model(selected_ckpt, custom_objects=custom_objects)
    probabilities = selected_model.predict(val_ds, verbose=0).reshape(-1)
    y_true = val_frame['label'].to_numpy(dtype=int)
    assert len(probabilities) == len(y_true) and np.isfinite(probabilities).all()

    predictions = val_frame[['relative_path', 'case_id', 'slide_id', 'label']].copy()
    predictions['probability'] = probabilities
    predictions['prediction'] = (probabilities >= 0.5).astype(int)
    predictions.to_csv(output_dir / f'{architecture}_{fold_name}_predictions.csv', index=False)

    case_rows = []
    for case_id, group in predictions.groupby('case_id'):
        true = group['label'].to_numpy(dtype=int)
        pred = group['prediction'].to_numpy(dtype=int)
        prob = group['probability'].to_numpy()
        case_rows.append({
            'case_id': case_id, 'patches': len(group),
            'accuracy': accuracy_score(true, pred), 'balanced_accuracy': balanced_accuracy_score(true, pred),
            'f1': f1_score(true, pred, zero_division=0),
            'roc_auc': roc_auc_score(true, prob) if len(np.unique(true)) == 2 else np.nan,
        })
    pd.DataFrame(case_rows).to_csv(output_dir / f'{architecture}_{fold_name}_case_metrics.csv', index=False)

    return {
        'fold_id': fold_id, 'selected_name': selected_name, 'selected_checkpoint': selected_ckpt,
        'custom_objects': custom_objects, 'validation_results': validation_results,
    }

## Run all 5 folds x 2 architectures

This is the compute-heavy cell. Each fold repeats exp-6/exp-7's full
two-phase protocol (frozen head, then fine-tuning) on ~25-26 training cases
and ~6-7 validation cases, so budget roughly the same per-fold wall-clock
time you already saw in exp-6 (ResNet50) and exp-7 (MobileNetV2) -- expect on
the order of a few hours total for all 10 runs on a Colab GPU runtime.

In [ ]:
fold_results: dict[str, dict[int, dict]] = {architecture: {} for architecture in ARCHITECTURE_BUILDERS}

for architecture in ARCHITECTURE_BUILDERS:
    for fold_id in range(5):
        fold_results[architecture][fold_id] = train_one_fold(architecture, fold_id, manifest, OUTPUT_DIR)

[resnet50/fold0] train cases=25 (16166 patches), val cases=7 (5635 patches)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5
506/506 - 103s - 204ms/step - accuracy: 0.9111 - loss: 0.2240 - precision: 0.9185 - recall: 0.9409 - roc_auc: 0.9668 - val_accuracy: 0.8774 - val_loss: 0.3738 - val_precision: 0.9200 - val_recall: 0.8780 - val_roc_auc: 0.9275 - learning_rate: 0.0010
Epoch 2/5
506/506 - 95s - 188ms/step - accuracy: 0.9526 - loss: 0.1365 - precision: 0.9554 - recall: 0.9692 - roc_auc: 0.9867 - val_accuracy: 0.8932 - val_loss: 0.3195 - val_precision: 0.9299 - val_recall: 0.8947 - val_roc_auc: 0.9448 - learning_rate: 0.0010
Epoch 3/5
506/506 - 143s - 282ms/step - accuracy: 0.9563 - loss: 0.1242 - precision: 0.9602 - recall: 0.9700 - roc_auc: 0.9887 - val_accuracy: 0.8882 - val_loss: 0.3966 - val_precision: 0.9418 - val_recall: 0.8732 - val_roc_auc: 0.9349 - learning_rate: 0.0010
Epoch 4/5
506/506 - 113s - 224ms/step - accuracy: 0.9591 - loss: 0.1166 - precision: 0.9637 - 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[resnet50/fold1] train cases=25 (16243 patches), val cases=7 (5558 patches)
Epoch 1/5
508/508 - 125s - 247ms/step - accuracy: 0.9211 - loss: 0.2081 - precision: 0.9260 - recall: 0.9485 - roc_auc: 0.9710 - val_accuracy: 0.9016 - val_loss: 0.2418 - val_precision: 0.8755 - val_recall: 0.9834 - val_roc_auc: 0.9775 - learning_rate: 0.0010
Epoch 2/5
508/508 - 139s - 274ms/step - accuracy: 0.9551 - loss: 0.1312 - precision: 0.9588 - recall: 0.9692 - roc_auc: 0.9874 - val_accuracy: 0.9066 - val_loss: 0.2298 - val_precision: 0.8846 - val_recall: 0.9794 - val_roc_auc: 0.9790 - learning_rate: 0.0010
Epoch 3/5
508/508 - 123s - 242ms/step - accuracy: 0.9585 - loss: 0.1181 - precision: 0.9622 - recall: 0.9712 - roc_auc: 0.9896 - val_accuracy: 0.8748 - val_loss: 0.3130 - val_precision: 0.8405 - val_recall: 0.9886 - val_roc_auc: 0.9777 - learning_rate: 0.0010
Epoch 4/5
508/508 - 142s - 279ms/step - accuracy: 0.9605 - loss: 0.1119 - precision: 0.9647 - recall: 0.9718 - roc_auc: 0.9906 - val_accuracy: 0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[resnet50/fold4] train cases=26 (18319 patches), val cases=6 (3482 patches)
Epoch 1/5
573/573 - 107s - 187ms/step - accuracy: 0.9289 - loss: 0.1829 - precision: 0.9341 - recall: 0.9446 - roc_auc: 0.9792 - val_accuracy: 0.8320 - val_loss: 0.4121 - val_precision: 0.9353 - val_recall: 0.8565 - val_roc_auc: 0.8479 - learning_rate: 0.0010
Epoch 2/5
573/573 - 103s - 179ms/step - accuracy: 0.9583 - loss: 0.1148 - precision: 0.9627 - recall: 0.9659 - roc_auc: 0.9913 - val_accuracy: 0.7702 - val_loss: 0.5167 - val_precision: 0.9449 - val_recall: 0.7674 - val_roc_auc: 0.8601 - learning_rate: 0.0010
Epoch 3/5
573/573 - 97s - 169ms/step - accuracy: 0.9609 - loss: 0.1041 - precision: 0.9651 - recall: 0.9679 - roc_auc: 0.9926 - val_accuracy: 0.8630 - val_loss: 0.4157 - val_precision: 0.9242 - val_recall: 0.9092 - val_roc_auc: 0.8257 - learning_rate: 0.0010
Epoch 4/5
573/573 - 97s - 169ms/step - accuracy: 0.9639 - loss: 0.0974 - precision: 0.9672 - recall: 0.9711 - roc_auc: 0.9935 - val_accuracy: 0.8

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[mobilenetv2/fold1] train cases=25 (16243 patches), val cases=7 (5558 patches)
Epoch 1/5
508/508 - 67s - 131ms/step - accuracy: 0.8891 - loss: 0.2741 - precision: 0.8935 - recall: 0.9321 - roc_auc: 0.9504 - val_accuracy: 0.9086 - val_loss: 0.2169 - val_precision: 0.9060 - val_recall: 0.9537 - val_roc_auc: 0.9706 - learning_rate: 0.0010
Epoch 2/5
508/508 - 81s - 159ms/step - accuracy: 0.9335 - loss: 0.1792 - precision: 0.9381 - recall: 0.9558 - roc_auc: 0.9782 - val_accuracy: 0.9163 - val_loss: 0.2015 - val_precision: 0.9214 - val_recall: 0.9480 - val_roc_auc: 0.9737 - learning_rate: 0.0010
Epoch 3/5
508/508 - 59s - 117ms/step - accuracy: 0.9419 - loss: 0.1659 - precision: 0.9468 - recall: 0.9603 - roc_auc: 0.9801 - val_accuracy: 0.9012 - val_loss: 0.2255 - val_precision: 0.8933 - val_recall: 0.9574 - val_roc_auc: 0.9716 - learning_rate: 0.0010
Epoch 4/5
508/508 - 82s - 160ms/step - accuracy: 0.9421 - loss: 0.1600 - precision: 0.9472 - recall: 0.9600 - roc_auc: 0.9815 - val_accuracy: 0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[mobilenetv2/fold4] train cases=26 (18319 patches), val cases=6 (3482 patches)
Epoch 1/5
573/573 - 60s - 105ms/step - accuracy: 0.9017 - loss: 0.2376 - precision: 0.9054 - recall: 0.9284 - roc_auc: 0.9646 - val_accuracy: 0.6872 - val_loss: 0.6215 - val_precision: 0.9276 - val_recall: 0.6752 - val_roc_auc: 0.8037 - learning_rate: 0.0010
Epoch 2/5
573/573 - 52s - 90ms/step - accuracy: 0.9406 - loss: 0.1560 - precision: 0.9441 - recall: 0.9547 - roc_auc: 0.9842 - val_accuracy: 0.7071 - val_loss: 0.6268 - val_precision: 0.9323 - val_recall: 0.6971 - val_roc_auc: 0.8094 - learning_rate: 0.0010
Epoch 3/5
573/573 - 83s - 145ms/step - accuracy: 0.9464 - loss: 0.1427 - precision: 0.9501 - recall: 0.9584 - roc_auc: 0.9866 - val_accuracy: 0.7961 - val_loss: 0.4961 - val_precision: 0.9355 - val_recall: 0.8097 - val_roc_auc: 0.8318 - learning_rate: 0.0010
Epoch 4/5
573/573 - 83s - 145ms/step - accuracy: 0.9503 - loss: 0.1369 - precision: 0.9540 - recall: 0.9611 - roc_auc: 0.9873 - val_accuracy: 0.7

## Final held-out test evaluation

Exactly one evaluation per architecture, on the 8-case test group, using the
fold-0 checkpoint prespecified above. Output schema matches exp-6/exp-7's
`*_test_metrics.json` and `*_test_predictions.csv` so `histopantum.metrics`
and `histopantum.data` work on these files unchanged.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_score, recall_score,
)

def evaluate_on_test(architecture: str, final: dict, manifest: pd.DataFrame, output_dir: Path) -> dict:
    test_frame = manifest.loc[manifest['group'] == 'test'].reset_index(drop=True)
    test_ds = make_dataset(test_frame, training=False)

    model = keras.models.load_model(final['selected_checkpoint'], custom_objects=final['custom_objects'])
    probabilities = model.predict(test_ds, verbose=0).reshape(-1)
    y_true = test_frame['label'].to_numpy(dtype=int)
    y_pred = (probabilities >= 0.5).astype(int)
    assert len(probabilities) == len(y_true) and np.isfinite(probabilities).all()

    predictions = test_frame[['relative_path', 'case_id', 'slide_id', 'label']].copy()
    predictions['probability'] = probabilities
    predictions['prediction'] = y_pred
    predictions.to_csv(output_dir / f'{architecture}_test_predictions.csv', index=False)

    pooled = {
        'accuracy': accuracy_score(y_true, y_pred), 'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0), 'roc_auc': roc_auc_score(y_true, probabilities),
        'patches': len(y_true), 'cases': int(predictions['case_id'].nunique()),
    }
    result = {
        'checkpoint': Path(final['selected_checkpoint']).name,
        'final_candidate_fold': FINAL_CANDIDATE_FOLD, 'threshold': 0.5,
        'pooled_patch_metrics': pooled, 'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(
            y_true, y_pred, target_names=['non-tumour', 'tumour'], output_dict=True, zero_division=0
        ),
    }
    with (output_dir / f'{architecture}_test_metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(result, handle, indent=2, allow_nan=False)
    print(f'[{architecture}] test accuracy={pooled["accuracy"]:.4f} roc_auc={pooled["roc_auc"]:.4f} '
          f'(cases={pooled["cases"]}, patches={pooled["patches"]})')
    return result


test_results = {}
for architecture in ARCHITECTURE_BUILDERS:
    final = fold_results[architecture][FINAL_CANDIDATE_FOLD]
    test_results[architecture] = evaluate_on_test(architecture, final, manifest, OUTPUT_DIR)

[resnet50] test accuracy=0.8979 roc_auc=0.9860 (cases=8, patches=5447)
[mobilenetv2] test accuracy=0.9189 roc_auc=0.9888 (cases=8, patches=5447)


## Preserve model identity and package compact evidence

In [ ]:
def sha256_file(path: Path) -> str:
    """Return the lowercase SHA-256 digest of a file."""
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

for architecture in ARCHITECTURE_BUILDERS:
    final = fold_results[architecture][FINAL_CANDIDATE_FOLD]
    checkpoint = Path(final['selected_checkpoint'])
    model_manifest = {
        'experiment_id': 'exp-8', 'architecture': architecture,
        'dataset': 'HISTOPANTUM colorectal subset', 'grouping_unit': 'TCGA case ID', 'seed': SEED,
        'holdout_split': '80/20 case-disjoint (32 cv-pool / 8 test), see cv_split_assignment.csv',
        'cross_validation': '5-fold StratifiedGroupKFold over the 80% cv-pool, grouped by case_id, '
                             'stratified by case tumour-fraction tertile',
        'final_candidate_fold': FINAL_CANDIDATE_FOLD,
        'selected_phase': final['selected_name'], 'model_file': checkpoint.name,
        'model_size_bytes': checkpoint.stat().st_size, 'model_sha256': sha256_file(checkpoint),
        'input_shape': [224, 224, 3], 'decision_threshold': 0.5,
        'limitations': [
            'The test evaluation uses fold 0\'s checkpoint only; the other 4 folds contribute '
            'to the cross-validation estimate but never touch the test cases.',
            'Only 40 TCGA cases are available in total across cv-pool and test.',
            'Patch-level observations within a case are correlated.',
        ],
    }
    with (OUTPUT_DIR / f'{architecture}_model_manifest.json').open('w', encoding='utf-8') as handle:
        json.dump(model_manifest, handle, indent=2)
    print(architecture, checkpoint.name, model_manifest['model_sha256'])

compact_dir = Path('/content/exp8_compact_files') if Path('/content').exists() else Path('exp8_compact_files')
if compact_dir.exists():
    shutil.rmtree(compact_dir)
compact_dir.mkdir(parents=True)
for artifact in OUTPUT_DIR.iterdir():
    if artifact.is_file() and artifact.suffix != '.keras':
        shutil.copy2(artifact, compact_dir / artifact.name)
archive_base = Path('/content/exp8_compact_evidence') if Path('/content').exists() else Path('exp8_compact_evidence')
archive = Path(shutil.make_archive(str(archive_base), 'zip', compact_dir))
print('Compact evidence (no .keras files):', archive, archive.stat().st_size, 'bytes')
print('Copy the compact evidence into experiments/exp-8/outputs/ in the repo, '
      'and download the fold-0 .keras checkpoints if you plan to run Phase-B-style '
      'inspection later.')

resnet50 resnet50_fold0_frozen.keras 9009e99dc43b011459726b3c43d7e2d0c3977213336e47e4fe9c1abbf6c1e0a8
mobilenetv2 mobilenetv2_fold0_fine_tuned.keras 296254d08d7ea48599facd4427a70625dfd61dc76feba3259822931d16ed4f7b
Compact evidence (no .keras files): /content/exp8_compact_evidence.zip 525295 bytes
Copy the compact evidence into experiments/exp-8/outputs/ in the repo, and download the fold-0 .keras checkpoints if you plan to run Phase-B-style inspection later.


In [ ]:
!zip -r /content/exp8_outputs.zip /content/exp8_outputs


  adding: content/exp8_outputs/ (stored 0%)
  adding: content/exp8_outputs/resnet50_fold0_fine_tuned.keras (deflated 7%)
  adding: content/exp8_outputs/resnet50_fold4_case_metrics.csv (deflated 45%)
  adding: content/exp8_outputs/resnet50_fold0_frozen.keras (deflated 8%)
  adding: content/exp8_outputs/resnet50_model_manifest.json (deflated 42%)
  adding: content/exp8_outputs/resnet50_test_predictions.csv (deflated 91%)
  adding: content/exp8_outputs/resnet50_fold4_predictions.csv (deflated 90%)
  adding: content/exp8_outputs/mobilenetv2_fold2_frozen.keras (deflated 12%)
  adding: content/exp8_outputs/resnet50_fold4_frozen.keras (deflated 8%)
  adding: content/exp8_outputs/resnet50_fold1_fine_tuned.keras (deflated 7%)
  adding: content/exp8_outputs/mobilenetv2_fold1_case_metrics.csv (deflated 45%)
  adding: content/exp8_outputs/mobilenetv2_fold1_fine_tuned.keras (deflated 9%)
  adding: content/exp8_outputs/resnet50_fold0_predictions.csv (deflated 91%)
  adding: content/exp8_outputs/resn